In [1]:
import hoda
import tensorly as tl
import os

%env TENSORLY_BACKEND=numpy
%env CUPY_CUDA_PER_THREAD_DEFAULT_STREAM=1


tl.set_backend(os.environ['TENSORLY_BACKEND'])
tl.get_backend()

env: TENSORLY_BACKEND=numpy
env: CUPY_CUDA_PER_THREAD_DEFAULT_STREAM=1


'numpy'

In [2]:
from moabb.paradigms import P300
from moabb.datasets import *


paradigm = P300(resample=48)
dataset = BNCI2014_008()
epochs, labels, meta = paradigm.get_data(
    dataset=dataset, 
     subjects=[1],
     return_epochs=True
)
session = meta['session'][0]
idc = meta['session'] == session
epochs = epochs[idc]
labels = labels[idc]
meta = meta[idc]

/data/leuven/352/vsc35289/miniconda3/envs/bttda/lib/python3.12/site-packages/moabb/datasets/preprocessing.py:279: UserWarning: warnEpochs <Epochs |  4200 events (all good), 0 – 1 s, baseline off, ~65.9 MB, data loaded,
 'Target': 700
 'NonTarget': 3500>
  warn(f"warnEpochs {epochs}")


Adding metadata with 3 columns
Adding metadata with 3 columns
4200 matching events found
No baseline correction applied


/data/leuven/352/vsc35289/miniconda3/envs/bttda/lib/python3.12/site-packages/moabb/paradigms/base.py:350: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  X = mne.concatenate_epochs(X)


In [3]:
import tensorly.decomposition
import matplotlib.pyplot as plt
import tensorly as tl

X = epochs.get_data(copy=True)
X = tl.tensor(X)
y = labels

In [4]:
from sklearn.model_selection import StratifiedKFold
from hoda.hoda import BTTDA, GreedyBTTDA, HODA, trunc_eigh
from hoda.cov import mode_scatter
from sklearn.pipeline import Pipeline
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from mne.decoding import Scaler
import  warnings
from sklearn.model_selection import GridSearchCV
from sklearn.feature_selection import SelectFwe
from sklearn.preprocessing import StandardScaler
import warnings
from joblib import parallel_backend
from joblib import Parallel

bttda = BTTDA(
    hoda_params=dict(
        rank=None,
        max_iter=64,
        tol=1e-8,
        init ='eye',
        shrinkage='lw',
        toeplitz=None,
        obj='rt',
        solver='lanczos',
        taper=False,
        extra_train_info=False,
        verbose=False,
        random_state=42,
        delta=None,
       
    ),
    verbose=False,
    extra_train_info=False,
)


In [5]:
import itertools
import numpy as np
B=3
ranks = [1,2,4,8]
ranks = list(itertools.product(*[ranks]*B))

In [6]:
ranks

[(1, 1, 1),
 (1, 1, 2),
 (1, 1, 4),
 (1, 1, 8),
 (1, 2, 1),
 (1, 2, 2),
 (1, 2, 4),
 (1, 2, 8),
 (1, 4, 1),
 (1, 4, 2),
 (1, 4, 4),
 (1, 4, 8),
 (1, 8, 1),
 (1, 8, 2),
 (1, 8, 4),
 (1, 8, 8),
 (2, 1, 1),
 (2, 1, 2),
 (2, 1, 4),
 (2, 1, 8),
 (2, 2, 1),
 (2, 2, 2),
 (2, 2, 4),
 (2, 2, 8),
 (2, 4, 1),
 (2, 4, 2),
 (2, 4, 4),
 (2, 4, 8),
 (2, 8, 1),
 (2, 8, 2),
 (2, 8, 4),
 (2, 8, 8),
 (4, 1, 1),
 (4, 1, 2),
 (4, 1, 4),
 (4, 1, 8),
 (4, 2, 1),
 (4, 2, 2),
 (4, 2, 4),
 (4, 2, 8),
 (4, 4, 1),
 (4, 4, 2),
 (4, 4, 4),
 (4, 4, 8),
 (4, 8, 1),
 (4, 8, 2),
 (4, 8, 4),
 (4, 8, 8),
 (8, 1, 1),
 (8, 1, 2),
 (8, 1, 4),
 (8, 1, 8),
 (8, 2, 1),
 (8, 2, 2),
 (8, 2, 4),
 (8, 2, 8),
 (8, 4, 1),
 (8, 4, 2),
 (8, 4, 4),
 (8, 4, 8),
 (8, 8, 1),
 (8, 8, 2),
 (8, 8, 4),
 (8, 8, 8)]

In [8]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
clf = Pipeline([
    ('bttda', bttda),
    ('vec', FunctionTransformer(tl.to_numpy)),
    ('zscore', StandardScaler()),
    ('lda', LinearDiscriminantAnalysis(shrinkage='auto', solver='lsqr'))
])

In [10]:
from sklearn.model_selection import GridSearchCV

param_grid = dict(bttda__ranks=ranks)
cv=StratifiedKFold(random_state=42, shuffle=True)

gs = GridSearchCV(clf, param_grid, scoring='roc_auc', n_jobs=8, cv=cv, verbose=True)
gs.fit(X,y)


Fitting 5 folds for each of 64 candidates, totalling 320 fits


GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('bttda',
                                        BTTDA(hoda_params={'delta': None,
                                                           'extra_train_info': False,
                                                           'init': 'eye',
                                                           'max_iter': 64,
                                                           'obj': 'rt',
                                                           'random_state': 42,
                                                           'rank': None,
                                                           'shrinkage': 'lw',
                                                           'solver': 'lanczos',
                                                           'taper': False,
                                                           'toeplitz': None,
                                                           'tol': 1e-08,
                                                           'verbose': False})),
                                       ('vec',
                                        FunctionTrans...
                                        LinearDiscriminantAnalysis(shrinkage='auto',
                                                                   solver='lsqr'))]),
             n_jobs=8,
             param_grid={'bttda__ranks': [(1, 1, 1), (1, 1, 2), (1, 1, 4),
                                          (1, 1, 8), (1, 2, 1), (1, 2, 2),
                                          (1, 2, 4), (1, 2, 8), (1, 4, 1),
                                          (1, 4, 2), (1, 4, 4), (1, 4, 8),
                                          (1, 8, 1), (1, 8, 2), (1, 8, 4),
                                          (1, 8, 8), (2, 1, 1), (2, 1, 2),
                                          (2, 1, 4), (2, 1, 8), (2, 2, 1),
                                          (2, 2, 2), (2, 2, 4), (2, 2, 8),
                                          (2, 4, 1), (2, 4, 2), (2, 4, 4),
                                          (2, 4, 8), (2, 8, 1), (2, 8, 2), ...]},
             scoring='roc_auc', verbose=True)

In [14]:
import pandas as pd

res = pd.DataFrame(gs.cv_results_)
res = res[res['params'] == dict(bttda__ranks=(8,2,1))]
res

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_bttda__ranks,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
52,3.800506,0.36516,0.009823,0.000162,"(8, 2, 1)","{'bttda__ranks': (8, 2, 1)}",0.866051,0.854571,0.815898,0.833969,0.832255,0.840549,0.017702,3
